# `extract_socioeconomico.ipynb` - Tabla maestra socioeconómica (Data Seeding)

Requisito: *"Tabla Maestra Socioeconómica (Data Seeding INE / Ayto): Inserción determinista de la renta media neta por persona (€) procedente del Atlas de Distribución de la Renta de los Hogares (INE) y la población empadronada por distrito (Ayuntamiento de Sevilla)."*

Los valores en sí (`POBLACION_DISTRITOS`, `RENTA_DISTRITOS`) ya están definidos como constantes en `config.ipynb` - son datos reales y verificados (ver la documentación de esa sección), no estimaciones. Este módulo simplemente los convierte en un `DataFrame` limpio, listo para cruzar con la geometría de los distritos.

> Depende de `config.ipynb`.

In [ ]:
import logging

import pandas as pd

## Reproducibilidad: de dónde salen los números

Por transparencia, así es exactamente como se obtuvo `RENTA_DISTRITOS` a partir del fichero oficial del INE (tabla `31205`, *Atlas de Distribución de Renta de los Hogares*, provincia de Sevilla) - el fichero está disponible en `data/31205.xlsx` para poder reproducirlo, pero no se ejecuta en cada corrida del pipeline (sería reprocesar un fichero nacional de miles de filas para obtener siempre las mismas 11 constantes), pero queda documentado aquí para que cualquiera pueda reproducirlo:

```python
import re
df_raw = pd.read_excel("./data/31205.xlsx", header=None)
patron_distrito = re.compile(r"^41091(0[1-9]|1[01]) Sevilla distrito (\d{2})$")
filas = [
    {"id_distrito": m.group(2), "renta": fila[1]}
    for _, fila in df_raw.iterrows()
    if (texto := str(fila[0]).strip()) and (m := patron_distrito.match(texto))
]
# columna 1 = "Renta neta media por persona", año 2023 (la más reciente)
```

`POBLACION_DISTRITOS` sale del *Informe Socioeconómico de la Ciudad de Sevilla 2023* (Ayuntamiento de Sevilla, Servicio de Estadística), tabla "Población por distritos" - los 11 valores suman exactamente 697.233 habitantes, el total oficial del Padrón Municipal.

## Función principal

In [ ]:
def extraer_tabla_socioeconomica():
    """
    Construye el DataFrame de la tabla maestra socioeconomica a partir
    de los diccionarios deterministas definidos en config.ipynb.
    """
    logger.info("Construyendo tabla maestra socioeconomica (data seeding INE / Ayuntamiento)...")

    nombres = sorted(POBLACION_DISTRITOS.keys())
    df = pd.DataFrame({
        "nombre_distrito": nombres,
        "poblacion_total": [POBLACION_DISTRITOS[n] for n in nombres],
        "renta_media_neta_persona": [RENTA_DISTRITOS[n] for n in nombres],
    })

    if set(POBLACION_DISTRITOS.keys()) != set(RENTA_DISTRITOS.keys()):
        logger.warning("Las claves de POBLACION_DISTRITOS y RENTA_DISTRITOS no coinciden exactamente.")

    logger.info(
        f"Tabla socioeconomica lista: {len(df)} distritos, "
        f"poblacion total {df['poblacion_total'].sum()} hab., "
        f"renta media {df['renta_media_neta_persona'].mean():.0f} EUR/persona (sin ponderar)."
    )
    return df

## Prueba rápida

In [ ]:
df_socio_prueba = extraer_tabla_socioeconomica()
print(df_socio_prueba.sort_values("renta_media_neta_persona", ascending=False).to_string(index=False))

---
✅ **Tabla maestra socioeconómica verificada**: 11 distritos, con población y renta reales.